# Isharah-500 data inspection

This notebook documents the real Isharah-500 keypoint contract used by the SI BiGRU + CTC baseline. It inspects only SI train and dev metadata and real tracked keypoint files. It never reads SI test labels.

The data root is supplied by `ISHARAH_DATA_ROOT` or by editing the configurable `DATA_ROOT` cell. The notebook reuses `isharah500/src/isharah_dataset.py`; it does not duplicate interpolation, validation, masking, or normalization logic.

In [ ]:
from pathlib import Path
import csv
import os
import sys

def resolve_project_root():
    configured = os.environ.get("ISHARAH_PROJECT_ROOT")
    candidates = [Path(configured)] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    for candidate in candidates:
        if (candidate / "isharah500" / "src").is_dir():
            return candidate
    raise RuntimeError("Set ISHARAH_PROJECT_ROOT to the repository worktree root.")

PROJECT_ROOT = resolve_project_root()
data_root_value = os.environ.get("ISHARAH_DATA_ROOT")
if not data_root_value:
    raise RuntimeError("Set ISHARAH_DATA_ROOT to the extracted real Isharah data root.")
DATA_ROOT = Path(data_root_value).resolve()
ISHARAH_ROOT = PROJECT_ROOT / "isharah500"
SOURCE_ROOT = ISHARAH_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))


from isharah_dataset import PreprocessingConfig, interpolate_keypoints, normalize_hand_centric, validate_keypoints

METADATA_DIR = ISHARAH_ROOT / "metadata"
TRACKED_ROOT = DATA_ROOT / "processed" / "isharah500" / "hand_keypoints_tracked"
print("Project:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Tracked root:", TRACKED_ROOT)

## Dataset and SI split

Isharah-500 contains continuous Saudi Sign Language samples represented as ordered frame sequences. The SI protocol uses 5,000 train samples, 500 dev samples, and 2,000 held-out test samples. This notebook uses only the train and dev CSV files for inspection.

Each tracked sample is expected to be a real `float32` array shaped `(T, 2, 63)`: two hand slots, 21 landmarks per hand, and XYZ coordinates per landmark.

In [ ]:
def read_metadata(split):
    if split not in {"train", "dev"}:
        raise ValueError("This inspection notebook intentionally excludes SI test metadata.")
    path = METADATA_DIR / f"isharah500_si_{split}.csv"
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        required = {"sample_id", "gloss", "text", "keypoints_path"}
        if not reader.fieldnames or not required.issubset(reader.fieldnames):
            raise ValueError(f"Unexpected metadata schema in {path}: {reader.fieldnames}")
        rows = list(reader)
    if not rows:
        raise ValueError(f"Metadata file is empty: {path}")
    return path, rows

train_path, train_rows = read_metadata("train")
dev_path, dev_rows = read_metadata("dev")
print("Train metadata:", train_path, len(train_rows))
print("Dev metadata:", dev_path, len(dev_rows))
print("First dev row:", dev_rows[0])

In [ ]:
def resolve_keypoints_path(row):
    candidate = DATA_ROOT / row["keypoints_path"]
    if candidate.is_file():
        return candidate
    parts = Path(row["keypoints_path"]).parts
    if parts[:2] == ("data", "processed"):
        candidate = DATA_ROOT / Path(*parts[2:])
    if not candidate.is_file():
        raise FileNotFoundError(f"Missing real keypoint file for {row['sample_id']}: {candidate}")
    return candidate

import numpy as np

def inspect_row(row):
    path = resolve_keypoints_path(row)
    keypoints = np.load(path, allow_pickle=False)
    validate_keypoints(keypoints, config=PreprocessingConfig(missing_file_policy="error"))
    return {"sample_id": row["sample_id"], "path": str(path), "shape": tuple(keypoints.shape), "dtype": str(keypoints.dtype), "nan_fraction": float(np.isnan(keypoints).mean()), "finite_fraction": float(np.isfinite(keypoints).mean())}

probe = [inspect_row(row) for row in (train_rows[:2] + dev_rows[:2])]
probe

## Missing detections, masks, interpolation, and normalization

Source NaNs represent missing landmark observations. The dataset contract derives a `(T, 2)` hand-detection mask before filling NaNs. Interpolation is temporal and is reused by inference. Hand-centric normalization subtracts each hand wrist and scales with the sequence median wrist-to-middle-finger-MCP distance. Degenerate or missing references are handled by the existing utility with finite fallbacks.

The following cell calls the production utilities directly on a real sample; it does not create replacement training data.

In [ ]:
config = PreprocessingConfig(nan_strategy="interpolate", flatten_hands=False, missing_file_policy="error", coordinate_normalization="hand_centric", normalization_epsilon=1e-6)
row = dev_rows[0]
raw = np.load(resolve_keypoints_path(row), allow_pickle=False)
validate_keypoints(raw, config=config)
valid_hand_mask = ~np.all(np.isnan(raw), axis=-1)
interpolated = interpolate_keypoints(raw)
normalized = normalize_hand_centric(interpolated, valid_hand_mask, epsilon=config.normalization_epsilon)
print({"sample_id": row["sample_id"], "raw_shape": raw.shape, "valid_hand_coverage": float(valid_hand_mask.mean()), "raw_nan_fraction": float(np.isnan(raw).mean()), "interpolated_finite": bool(np.isfinite(interpolated).all()), "normalized_finite": bool(np.isfinite(normalized).all())})

## Optional visualization

The next cell is read-only and requires a local real dev keypoint file. It visualizes one coordinate trace before and after the saved preprocessing. It does not write an image or inspect test data.

In [ ]:
# Optional visualization cell; execute only after the real-data paths above are configured.
import matplotlib.pyplot as plt
coordinate = 0
plt.figure(figsize=(10, 4))
plt.plot(raw[:, 0, coordinate], label="raw wrist x", alpha=0.6)
plt.plot(normalized[:, 0, coordinate], label="normalized wrist x")
plt.title(f"SI dev sample {row['sample_id']}: coordinate trace")
plt.xlabel("frame")
plt.ylabel("value")
plt.legend()
plt.show()